In [5]:
import os
import sys

project_root = os.path.abspath(os.path.join(os.getcwd(), "../.."))

if project_root not in sys.path:
    sys.path.append(project_root)

In [6]:
import pipeline.src.python.config as cfg
import pandas as pd
import numpy as np
pd.set_option('display.max_colwidth', None)

In [9]:
import networkx as nx

In [8]:
MAGAZINE_1 = 'scopus'
DATASET_TEXT_FEATURE = (
    "text"  # In the dataset file, the column name that contains the text data
)
cfg_dict_1 = cfg.MAGAZINE_CONFIG[MAGAZINE_1]

In [9]:
MAGAZINE_2 = 'the_guardian'
cfg_dict_2 = cfg.MAGAZINE_CONFIG[MAGAZINE_2]

In [10]:
MAGAZINE_3 = 'science_news'
cfg_dict_3 = cfg.MAGAZINE_CONFIG[MAGAZINE_3]

In [11]:
from bertopic import BERTopic

model_path = cfg.MODELS_FOLDER / f'{MAGAZINE_1}/model_0.343.safetensors'
model_1 = BERTopic.load(model_path, 
                      embedding_model=cfg.EMBEDDING_MODEL
                      )

/home/banfi/.uve/cuda/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [12]:
model_path_2 = cfg.MODELS_FOLDER / f'{MAGAZINE_2}/model_0.334.safetensors'

model_2 = BERTopic.load(model_path_2,
                        cfg.EMBEDDING_MODEL
                        )


In [13]:
model_path_3 = cfg.MODELS_FOLDER / f'{MAGAZINE_3}/model_0.309.safetensors'

model_3 = BERTopic.load(model_path_3,
                        cfg.EMBEDDING_MODEL
                        )


In [15]:
data = np.load(cfg_dict_1['OUTPUT_PATH'],allow_pickle=True) 

ids = data['id']
texts = data['text'] 
embeddings = data['embedding'] 
documents = data['clean_text']

In [16]:
data_2 = np.load('/home/banfi/TETYS/pipeline/src/python/data/interim/embeddings/the_guardian/the_guardian_embeddings_summary_filtered.npz',allow_pickle=True) 

ids_2 = data_2['id']
texts_2 = data_2['text'] 
embeddings_2 = data_2['embedding'] 
documents_2 = data_2['clean_text']

In [17]:
data_3 = np.load(cfg_dict_3['OUTPUT_PATH'],allow_pickle=True) 

ids_3 = data_3['id']
texts_3 = data_3['text'] 
embeddings_3 = data_3['embedding'] 
documents_3 = data_3['clean_text']

### Model 1 vs Model 2

In [ ]:
from pipeline.src.python.btm import BTM

models_metrics_1_vs_2 = BTM(model_1=model_1,
                            model_2=model_2,
                            ids_1=ids,
                            texts_1=texts,
                            embeddings_1=embeddings,
                            texts_2=texts_2,
                            embeddings_2=embeddings_2,
                            model_1_name=MAGAZINE_1.title(),
                            model_2_name=MAGAZINE_2.title())

2026-02-09 14:09:37,164 - BERTopic - Predicting topic assignments through cosine similarity of topic and document embeddings.
2026-02-09 14:09:37,370 - BERTopic - Predicting topic assignments through cosine similarity of topic and document embeddings.


Threshold:0.3820500373840332


In [25]:
models_metrics_1_vs_2.evaluate_metrics()

In [26]:
model_1_vs_model_2_closeness = models_metrics_1_vs_2.get_topic_closeness().sort_values(by='Topic Closeness',ascending=False)

In [ ]:
inverse_models_metrics_2_vs_1 = BTM(model_1=model_2,
                                    model_2=model_1,
                                    ids_1=ids_2,
                                    texts_1=texts_2,
                                    embeddings_1=embeddings_2,
                                    texts_2=texts,
                                    embeddings_2=embeddings,
                                    model_1_name=MAGAZINE_2.title(),model_2_name=MAGAZINE_1.title())

2026-02-09 14:09:40,871 - BERTopic - Predicting topic assignments through cosine similarity of topic and document embeddings.
2026-02-09 14:09:42,092 - BERTopic - Predicting topic assignments through cosine similarity of topic and document embeddings.


Threshold:0.38246655464172363


In [28]:
inverse_models_metrics_2_vs_1.evaluate_metrics()

In [29]:
model_2_vs_model_1_closeness = inverse_models_metrics_2_vs_1.get_topic_closeness().sort_values(by='Topic Closeness',ascending=False)

### Model 1 vs Model 3

In [ ]:
from pipeline.src.python.btm import BTM

models_metrics_1_vs_3 = BTM(model_1=model_1,
                            model_2=model_3,
                            ids_1=ids,
                            texts_1=texts,
                            embeddings_1=embeddings,
                            texts_2=texts_3,
                            embeddings_2=embeddings_3,
                            model_1_name=MAGAZINE_1.title(),
                            model_2_name=MAGAZINE_3.title())

2026-02-09 14:09:54,709 - BERTopic - Predicting topic assignments through cosine similarity of topic and document embeddings.
2026-02-09 14:09:54,721 - BERTopic - Predicting topic assignments through cosine similarity of topic and document embeddings.


Threshold:0.3790895342826843


In [31]:
models_metrics_1_vs_3.evaluate_metrics()

In [32]:
model_1_vs_model_3_closeness = models_metrics_1_vs_3.get_topic_closeness().sort_values(by='Topic Closeness',ascending=False)

In [ ]:
inverse_models_metrics_3_vs_1 = BTM(model_1=model_3,
                                    model_2=model_1,
                                    ids_1=ids_3,
                                    texts_1=texts_3,
                                    embeddings_1=embeddings_3,
                                    texts_2=texts,
                                    embeddings_2=embeddings,
                                    model_1_name=MAGAZINE_3.title(),
                                    model_2_name=MAGAZINE_1.title())

2026-02-09 14:11:39,363 - BERTopic - Predicting topic assignments through cosine similarity of topic and document embeddings.
2026-02-09 14:11:40,577 - BERTopic - Predicting topic assignments through cosine similarity of topic and document embeddings.


Threshold:0.38246655464172363


In [34]:
inverse_models_metrics_3_vs_1.evaluate_metrics()

In [35]:
model_3_vs_model_1_closeness = inverse_models_metrics_3_vs_1.get_topic_closeness().sort_values(by='Topic Closeness',ascending=False)

### Model 2 vs Model 3

In [36]:
from pipeline.src.python.btm import BTM

models_metrics_2_vs_3 = BTM(model_1=model_2,
                            model_2=model_3,
                            ids_1=ids_2,
                            texts_1=texts_2,
                            embeddings_1=embeddings_2,
                            texts_2=texts_3,
                            embeddings_2=embeddings_3,
                            model_1_name=MAGAZINE_2.title(),
                            model_2_name=MAGAZINE_3.title()
                            )

2026-02-09 14:14:37,245 - BERTopic - Predicting topic assignments through cosine similarity of topic and document embeddings.
2026-02-09 14:14:37,257 - BERTopic - Predicting topic assignments through cosine similarity of topic and document embeddings.


Threshold:0.3790895342826843


In [37]:
models_metrics_2_vs_3.evaluate_metrics()

In [38]:
model_2_vs_model_3_closeness = models_metrics_2_vs_3.get_topic_closeness().sort_values(by='Topic Closeness',ascending=False)

In [39]:
inverse_models_metrics_3_vs_2 = BTM(model_1=model_3,
                                    model_2=model_2,
                                    ids_1=ids_3,
                                    texts_1=texts_3,
                                    embeddings_1=embeddings_3,
                                    texts_2=texts_2,
                                    embeddings_2=embeddings_2,
                                    model_1_name=MAGAZINE_3.title(),
                                    model_2_name=MAGAZINE_2.title())

2026-02-09 14:16:10,197 - BERTopic - Predicting topic assignments through cosine similarity of topic and document embeddings.
2026-02-09 14:16:10,400 - BERTopic - Predicting topic assignments through cosine similarity of topic and document embeddings.


Threshold:0.3820500373840332


In [41]:
inverse_models_metrics_3_vs_2.evaluate_metrics()

In [40]:
model_3_vs_model_2_closeness = inverse_models_metrics_3_vs_2.get_topic_closeness().sort_values(by='Topic Closeness',ascending=False)

## Renaming

In [46]:
model_1_vs_model_2_closeness = model_1_vs_model_2_closeness.rename(columns={'Scopus Topic Label':'Model 1 Label',
                                                                            'The_Guardian Topic Label':'Model 2 Label'})[['Model 1 Label','Model 2 Label','Topic Closeness']]

In [47]:
model_2_vs_model_1_closeness = model_2_vs_model_1_closeness.rename(columns={'Scopus Topic Label':'Model 2 Label',
                                                                            'The_Guardian Topic Label':'Model 1 Label'})[['Model 1 Label','Model 2 Label','Topic Closeness']]

In [49]:
model_1_vs_model_3_closeness = model_1_vs_model_3_closeness.rename(columns={'Scopus Topic Label':'Model 1 Label',
                                                                            'Science_News Topic Label':'Model 2 Label'})[['Model 1 Label','Model 2 Label','Topic Closeness']]

In [50]:
model_3_vs_model_1_closeness = model_3_vs_model_1_closeness.rename(columns={'Scopus Topic Label':'Model 2 Label',
                                                                            'Science_News Topic Label':'Model 1 Label'})[['Model 1 Label','Model 2 Label','Topic Closeness']]

In [52]:
model_2_vs_model_3_closeness = model_2_vs_model_3_closeness.rename(columns={'The_Guardian Topic Label':'Model 1 Label',
                                                                            'Science_News Topic Label':'Model 2 Label'})[['Model 1 Label','Model 2 Label','Topic Closeness']]

In [53]:
model_3_vs_model_2_closeness = model_3_vs_model_2_closeness.rename(columns={'The_Guardian Topic Label':'Model 2 Label',
                                                                            'Science_News Topic Label':'Model 1 Label'})[['Model 1 Label','Model 2 Label','Topic Closeness']]

### Creates dataset

In [58]:
closeness = pd.concat([model_1_vs_model_2_closeness,
                model_2_vs_model_1_closeness,
                model_1_vs_model_3_closeness,
                model_3_vs_model_1_closeness,
                model_2_vs_model_3_closeness,
                model_3_vs_model_2_closeness])


In [59]:
len_tot = ( len(model_1_vs_model_2_closeness) +
            len(model_2_vs_model_1_closeness) +
            len(model_1_vs_model_3_closeness) +
            len(model_3_vs_model_1_closeness) +
            len(model_2_vs_model_3_closeness) +
            len(model_3_vs_model_2_closeness)
)

In [60]:
len_tot

7600

In [63]:
closeness.to_parquet('edges.parquet')

In [4]:
edges

,Model 1 Label,Model 2 Label,Topic Closeness
159,Post-Acute COVID-19 Syndrome and Long COVID,Long-term Covid Condition,0.991189
402,Prison Health and Safety,Prison System Challenges,0.988235
361,Pregnancy and Influenza Vaccination,Pregnant Individuals and Vaccination,0.970297
92,Animal Antimicrobial Use,Antibiotic Use in Livestock Farming,0.963446
69,Zika Virus Outbreaks,Zika Virus Outbreak Risk,0.963415
...,...,...,...
103,Covid-19 Vaccine Availability and Effectiveness,Vaccine Policy and Misinformation,0.009804
106,Covid-19 Vaccine Availability and Effectiveness,Measles Vaccination Efforts,0.009804
105,Covid-19 Vaccine Availability and Effectiveness,Misinformation Spread During Pandemic,0.009804
114,Covid-19 Vaccine Availability and Effectiveness,Russia-Ukraine Tensions Escalate,0.009804


In [26]:
counts = model_1.get_topic_freq()['Count'][1:].to_list() if -1 in model_1.topics_ else model_1.get_topic_freq()['Count'].to_list()
labels  = model_1.custom_labels_[1:] if -1 in model_1.topics_ else model_1.custom_labels_

model_1_nodes = pd.DataFrame({'Topic Label': labels,
                              'Degree':counts })
model_1_nodes['Model'] = MAGAZINE_1.title()


In [28]:
counts_2 = model_2.get_topic_freq()['Count'][1:].to_list() if -1 in model_2.topics_ else model_2.get_topic_freq()['Count'].to_list()
labels_2  = model_2.custom_labels_[1:] if -1 in model_2.topics_ else model_2.custom_labels_

model_2_nodes = pd.DataFrame({'Topic Label': labels_2,
                              'Degree':counts_2 })
model_2_nodes['Model'] = MAGAZINE_2.title()

In [30]:
counts_3 = model_3.get_topic_freq()['Count'][1:].to_list() if -1 in model_3.topics_ else model_3.get_topic_freq()['Count'].to_list()
labels_3  = model_3.custom_labels_[1:] if -1 in model_3.topics_ else model_3.custom_labels_

model_3_nodes = pd.DataFrame({'Topic Label': labels_3,
                              'Degree':counts_3 })
model_3_nodes['Model'] = MAGAZINE_3.title()

In [32]:
nodes = pd.concat([model_1_nodes,
                model_2_nodes,
                model_3_nodes])

In [33]:
nodes.to_parquet('nodes.parquet')

## Retriving data

In [236]:
import pandas as pd
edges = pd.read_parquet('edges.parquet')
nodes = pd.read_parquet('nodes.parquet')

In [237]:
edges_to_be_filtered = edges.merge(nodes,how='inner',left_on='Model 1 Label',right_on='Topic Label')

In [239]:
THRESHOLD_SN = 5
THRESHOLD_TG = 20
THRESHOLD_SCOPUS = 25

In [240]:
edges_to_be_filtered['N_Match'] = edges_to_be_filtered['Topic Closeness']  * edges_to_be_filtered['Degree']

In [241]:
edges_sn_filtered = edges_to_be_filtered[ (edges_to_be_filtered['Model'] == 'Science_News') & (edges_to_be_filtered['N_Match'] > THRESHOLD_SN)]

In [242]:
edges_scopus_filtered = edges_to_be_filtered[ (edges_to_be_filtered['Model'] == 'Scopus') & (edges_to_be_filtered['N_Match'] > THRESHOLD_SCOPUS)]

In [243]:
edges_tg_filtered = edges_to_be_filtered[ (edges_to_be_filtered['Model'] == 'The_Guardian') & (edges_to_be_filtered['N_Match'] > THRESHOLD_TG)]

In [244]:
edges_filtered = pd.concat([edges_sn_filtered,edges_scopus_filtered,edges_tg_filtered])

In [245]:
edges_filtered = edges_filtered[['Model 1 Label','Model 2 Label','Topic Closeness']]

In [246]:

G = nx.DiGraph(description='Connected components') 

for _, node in nodes.iterrows():
    G.add_node(
    node['Topic Label'],
    degree=node['Degree'],
    model=node['Model']
)

for _, edge in edges_filtered.iterrows():
    G.add_edge(
    edge['Model 1 Label'],
    edge['Model 2 Label'],
    weight=edge['Topic Closeness']
)



In [247]:
G_und = nx.Graph()

for u, v, data in G.edges(data=True):
    w = data["weight"]

    if G_und.has_edge(u, v):
        G_und[u][v]["weight"] += w
    else:
        G_und.add_edge(u, v, weight=w)

## Louvain

In [248]:
import community as community_louvain

In [249]:
partition = community_louvain.best_partition(G_und)

In [250]:
distinct_partition = []
n_partition = [ distinct_partition.append(v) for k,v in partition.items() if v not in distinct_partition]

In [251]:
len(distinct_partition)

26

In [252]:
from collections import defaultdict

clusters = defaultdict(list)

for node, comm in partition.items():
    clusters[comm].append(node)

for c, nodes in clusters.items():
    print(f"Cluster {c}:")
    print(", ".join(nodes))
    print()


Cluster 25:
High Sensitivity Biosensor for Bacterial Cell Detection, Disease Outbreak Tracking Initiative, Water Quality Microbial Contamination, Genetic Research and Data Insights, Wastewater Surveillance for Cov-2 Detection, Covid Testing Kits, Global Misinformation Concern, Exome Variant Analysis with HaplotypeCaller, Wearable Health Monitoring System, Contact Tracing Apps and Concerns, Metagenomic Pathogen Genomics, Covid-19 Vaccine Misinformation Analysis, Misinformation Spread During Pandemic, Outbreak Detection Surveillance System, Genetic Disease and Variant Analysis, Covid-19 Contact Tracing App, Surveillance and Reform Concerns, AI-Powered Pandemic Surveillance, Covid-19 Diagnostic Assay, Influenza Surveillance Systems, Covid-19 Search Trends and Correlation Analysis, Global Biosurveillance Strategy, Covid-19 Antigen Rapid Testing, Covid-19 Testing Shortage in Australia, SARS-CoV-2 Detection Methods, Saliva-Based Covid-19 Detection, Social Distancing Monitoring System, Face D

In [170]:
label_name = []
cluster_number = []
for k,v in partition.items():
    label_name.append(k)
    cluster_number.append(v)

In [171]:
louvain_cluster = pd.DataFrame({'Topic Label':label_name,
              'Cluster': cluster_number})

In [172]:
louvain_cluster.to_csv('louvain_cluster.csv',sep='ç',encoding='utf-8')

## Leiden

In [253]:
import leidenalg
import igraph as ig

In [254]:
nodes = list(G_und.nodes())
G_ig = ig.Graph.from_networkx(G_und)
G_ig.vs["name"] = nodes

In [255]:
partition = leidenalg.find_partition(
    G_ig,
    leidenalg.ModularityVertexPartition,
    weights="weight",
)

In [256]:
clusters = partition.membership


In [257]:
node_to_cluster = {
    v["name"]: clusters[i]
    for i, v in enumerate(G_ig.vs)
}


In [258]:
from collections import defaultdict

clusters = defaultdict(list)

for node, comm in node_to_cluster.items():
    clusters[comm].append(node)

for c, nodes in clusters.items():
    print(f"Cluster {c}:")
    print(", ".join(nodes))
    print()


Cluster 1:
High Sensitivity Biosensor for Bacterial Cell Detection, Antibiotic Resistance Threats, Bacterial Infection Mechanisms, Antibiotic Resistance Crisis, Effective Antibiotics Against Resistant Bacteria, Antibiotic Use in Livestock Farming, Antibacterial Activity of Silver Nanoparticles, Pneumococcal Vaccine Serotypes and Coverage, Hypervirulent Klebsiella Pneumoniae, Antibiotic Resistance in Wastewater Treatment, Antimicrobial Resistance in Acinetobacter baumannii, Fish Disease and Pathogen Impact, Campylobacter Infections and Prevalence, Carbapenem-Resistant Aeruginosa Strains, Carbapenem Resistance and Carbapenemase-Producing Bacteria, Antimicrobial Resistance in Neisseria gonorrhoeae, ESBL-Producing Enterobacteriaceae Infections, MCR-1 Gene and Colistin Resistance Prevalence, Antimicrobial Resistance Management, Prevalence of Antibiotic-Resistant Uropathogens in UT, Antibiotic Use and Prescribing Trends, Animal Antimicrobial Use, Antimicrobial Stewardship in Hospitals, Antim

In [131]:
label_name = []
cluster_number = []
for k,v in node_to_cluster.items():
    label_name.append(k)
    cluster_number.append(v)


In [132]:
leiden_cluster = pd.DataFrame({'Topic Label':label_name,
              'Cluster': cluster_number})

In [133]:
leiden_cluster.to_csv('leiden_cluster.csv',sep='ç',encoding='utf-8')